# Synthetic warehouse images on Nebius

This notebook drives the whole loop from your machine: check the setup, probe a GPU job, launch a run, and look at what it produced. The GPU work happens in Nebius serverless jobs; this notebook only launches them and reads the bucket.

One-time setup is in `docs/NEBIUS_SETUP.md`.

## 0. Check this machine
Checks the Nebius CLI, Docker and its registry login, `.env`, the shared filesystem, and bucket access with your keys. No GPU job is started.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Image, display

load_dotenv(".env")

import nebius_job as nj
from sdg import storage
from sdg.config import load_config

assert nj.check(), "fix the FAILs above (see docs/NEBIUS_SETUP.md)"

## 1. Dry run
Prints what a launch would do: build and push the job image (tagged with a hash of the code, so this is quick when nothing changed), then the `nebius ai job create` command.

In [ ]:
nj.launch("configs/naive_20.json", dry_run=True)

## 2. Probe a GPU job
The probe checks the job side: GPU graphics support, Vulkan, the shared cache, network, and the bucket keys from MysteryBox. Then it renders the warehouse in four anti-aliasing modes. The first probe is a cold start of about 6 minutes, and it warms the caches for later jobs.

`nj.launch()` builds and pushes the image first. The first push uploads the ~11 GB Isaac base layers once.

In [ ]:
probe_id = nj.launch("configs/probe.json")
nj.wait(probe_id)
nj.logs(probe_id, tail=40)

Each mode's frame, with its scores. A clean frame has an `edge` score of about 1–2.

In [ ]:
probe_cfg = load_config("configs/probe.json")
job_name = nj.last_job("probe")["name"]
probe_dir = storage.download(probe_cfg, f"{storage.run_prefix(probe_cfg)}/{job_name}", Path("runs/probe") / job_name)
print((probe_dir / "result.txt").read_text())
for png in sorted(probe_dir.glob("*.png")):
    print(png.stem)
    display(Image(filename=str(png), width=960))

## 3. The naive run
20 frames per camera at the stock example's density: a handful of objects, so most frames are near-empty. The gate is expected to fail on density, and the job still ends COMPLETED because `expect_gate` is `"fail"`.

In [ ]:
naive_id = nj.launch("configs/naive_20.json")
nj.wait(naive_id)
nj.logs(naive_id, tail=40)

## 4. Audit the run
Downloads the run to `runs/naive_20/`, prints the gate's report, and shows a contact sheet per camera with the boxes drawn on.

In [ ]:
cfg = load_config("configs/naive_20.json")
run_dir = storage.download(cfg, storage.run_prefix(cfg), Path("runs") / cfg["run_name"])
print((run_dir / "validation.txt").read_text())
for sheet in sorted(run_dir.glob("contact_*.jpg")):
    print(sheet.stem)
    display(Image(filename=str(sheet), width=960))